# AEGIS Routing System Test Suite

This notebook contains comprehensive test cases to evaluate the AEGIS agentic routing system.
We test:
1. Router decisions (research vs direct response)
2. Database selection logic
3. Edge cases and error handling
4. Multi-turn conversation handling

In [ ]:
import sys
import os
from pathlib import Path
import json
from datetime import datetime
from typing import Dict, List, Tuple, Optional
import pandas as pd
from IPython.display import display, HTML

# Add project root to path
project_root = Path(os.getcwd()).parent
sys.path.insert(0, str(project_root))

# Import AEGIS components
from aegis.src.conversation_setup.conversation import ConversationManager
from aegis.src.agents.agent_router.router import RouterAgent
from aegis.src.agents.agent_clarifier.clarifier import ClarifierAgent
from aegis.src.agents.agent_planner.planner import PlannerAgent
from aegis.src.agents.database_subagents.database_router import DatabaseRouter
from aegis.src.agents.agent_direct_response.response_from_conversation import DirectResponseAgent
from aegis.src.chat_model.model import ChatModel

## Test Case Definitions

In [ ]:
# Define comprehensive test cases
TEST_CASES = [
    # Financial Data Requests (Should route to RESEARCH)
    {
        "category": "Financial Data",
        "query": "What was RBC's revenue in Q3 2024?",
        "expected_route": "research",
        "expected_databases": ["public_rts", "public_benchmarking"],
        "reasoning": "Specific financial metric request"
    },
    {
        "category": "Financial Data",
        "query": "Show me TD Bank's net interest margin trends over the last 4 quarters",
        "expected_route": "research",
        "expected_databases": ["public_benchmarking", "public_rts"],
        "reasoning": "Time series financial data request"
    },
    {
        "category": "Financial Data",
        "query": "Compare BMO's and Scotiabank's efficiency ratios",
        "expected_route": "research",
        "expected_databases": ["public_benchmarking", "public_rts"],
        "reasoning": "Comparative financial metrics"
    },
    
    # Management Commentary (Should route to RESEARCH)
    {
        "category": "Management Commentary",
        "query": "What did RBC's CEO say about digital transformation in the latest earnings call?",
        "expected_route": "research",
        "expected_databases": ["public_transcripts"],
        "reasoning": "Executive commentary request"
    },
    {
        "category": "Management Commentary",
        "query": "What guidance did TD provide for 2025?",
        "expected_route": "research",
        "expected_databases": ["public_transcripts", "public_rts"],
        "reasoning": "Forward guidance request"
    },
    {
        "category": "Management Commentary",
        "query": "How are Canadian banks managing credit risk in the current environment?",
        "expected_route": "research",
        "expected_databases": ["public_transcripts", "public_rts"],
        "reasoning": "Management strategy and commentary"
    },
    
    # General Questions (Should route to DIRECT RESPONSE)
    {
        "category": "General Concepts",
        "query": "What is net interest margin?",
        "expected_route": "direct_response",
        "expected_databases": [],
        "reasoning": "Basic financial concept explanation"
    },
    {
        "category": "General Concepts",
        "query": "How do Canadian banks differ from US banks?",
        "expected_route": "direct_response",
        "expected_databases": [],
        "reasoning": "General industry knowledge"
    },
    
    # Follow-up Questions (Context-dependent)
    {
        "category": "Follow-up",
        "query": "Can you explain that in simpler terms?",
        "expected_route": "direct_response",
        "expected_databases": [],
        "reasoning": "Clarification of existing information",
        "requires_context": True
    },
    {
        "category": "Follow-up",
        "query": "What about their international operations?",
        "expected_route": "research",
        "expected_databases": ["public_transcripts", "public_rts"],
        "reasoning": "New aspect requiring research",
        "requires_context": True
    },
    
    # Edge Cases
    {
        "category": "Edge Case",
        "query": "Tell me everything about RBC",
        "expected_route": "direct_response",
        "expected_databases": [],
        "reasoning": "Too broad, needs clarification"
    },
    {
        "category": "Edge Case",
        "query": "What's the weather like?",
        "expected_route": "direct_response",
        "expected_databases": [],
        "reasoning": "Off-topic query"
    },
    
    # Complex Queries
    {
        "category": "Complex",
        "query": "How has RBC's digital banking revenue grown compared to traditional banking, and what are executives saying about future investments?",
        "expected_route": "research",
        "expected_databases": ["public_transcripts", "public_rts", "public_benchmarking"],
        "reasoning": "Multi-faceted query requiring multiple data sources"
    },
    {
        "category": "Complex",
        "query": "Which Canadian bank has the best ROE and what's driving their performance?",
        "expected_route": "research",
        "expected_databases": ["public_benchmarking", "public_transcripts", "public_rts"],
        "reasoning": "Comparative analysis with causal factors"
    },
    
    # Specific Database Tests
    {
        "category": "Database Selection",
        "query": "What are the key takeaways from BMO's Q2 2024 earnings call?",
        "expected_route": "research",
        "expected_databases": ["public_transcripts"],
        "reasoning": "Earnings call specific - transcripts only"
    },
    {
        "category": "Database Selection",
        "query": "Show me the detailed breakdown of Scotia's loan portfolio",
        "expected_route": "research",
        "expected_databases": ["public_rts"],
        "reasoning": "Detailed financial statement data"
    },
    {
        "category": "Database Selection",
        "query": "Compare Canadian banks' CET1 ratios",
        "expected_route": "research",
        "expected_databases": ["public_benchmarking"],
        "reasoning": "Standardized metrics comparison"
    }
]

## Test Execution Framework

In [ ]:
class AegisTestRunner:
    """Test runner for AEGIS routing system"""
    
    def __init__(self):
        self.conversation_manager = ConversationManager()
        self.router_agent = RouterAgent()
        self.clarifier_agent = ClarifierAgent()
        self.planner_agent = PlannerAgent()
        self.database_router = DatabaseRouter()
        self.direct_response_agent = DirectResponseAgent()
        self.results = []
        
    def run_single_test(self, test_case: Dict, context: Optional[str] = None) -> Dict:
        """Run a single test case and return results"""
        result = test_case.copy()
        result['timestamp'] = datetime.now().isoformat()
        
        try:
            # Update conversation with query
            if context:
                self.conversation_manager.add_message("assistant", context)
            self.conversation_manager.add_message("user", test_case['query'])
            
            # Step 1: Clarifier
            clarifier_result = self.clarifier_agent.clarify_query(
                test_case['query'],
                self.conversation_manager.get_conversation_history()
            )
            result['clarified_query'] = clarifier_result.get('research_statement', '')
            
            # Step 2: Router decision
            router_result = self.router_agent.route_query(
                test_case['query'],
                self.conversation_manager.get_conversation_history()
            )
            result['actual_route'] = router_result.get('route', 'unknown')
            result['router_reasoning'] = router_result.get('reasoning', '')
            
            # Step 3: If research route, check database selection
            if result['actual_route'] == 'research':
                planner_result = self.planner_agent.plan_research(
                    clarifier_result.get('research_statement', test_case['query']),
                    self.conversation_manager.get_conversation_history()
                )
                result['actual_databases'] = planner_result.get('databases', [])
                result['planner_reasoning'] = planner_result.get('reasoning', '')
            else:
                result['actual_databases'] = []
                
            # Evaluate success
            result['route_correct'] = result['actual_route'] == test_case['expected_route']
            
            if test_case['expected_databases']:
                expected_set = set(test_case['expected_databases'])
                actual_set = set(result['actual_databases'])
                result['databases_correct'] = len(expected_set.intersection(actual_set)) > 0
            else:
                result['databases_correct'] = len(result['actual_databases']) == 0
                
            result['success'] = result['route_correct']
            
        except Exception as e:
            result['error'] = str(e)
            result['success'] = False
            result['route_correct'] = False
            result['databases_correct'] = False
            
        return result
    
    def run_all_tests(self, test_cases: List[Dict]) -> pd.DataFrame:
        """Run all test cases and return results as DataFrame"""
        self.results = []
        
        for i, test_case in enumerate(test_cases):
            print(f"Running test {i+1}/{len(test_cases)}: {test_case['query'][:50]}...")
            
            # Reset conversation for non-follow-up questions
            if not test_case.get('requires_context', False):
                self.conversation_manager = ConversationManager()
            
            # Add context for follow-up questions
            context = None
            if test_case.get('requires_context', False):
                context = "RBC reported revenue of $12.5 billion in Q3 2024, with strong performance in capital markets."
                
            result = self.run_single_test(test_case, context)
            self.results.append(result)
            
        return pd.DataFrame(self.results)
    
    def generate_report(self, df: pd.DataFrame) -> None:
        """Generate test report with visualizations"""
        print("\n" + "="*80)
        print("AEGIS ROUTING SYSTEM TEST REPORT")
        print("="*80 + "\n")
        
        # Overall statistics
        total_tests = len(df)
        route_success = df['route_correct'].sum()
        db_success = df[df['expected_route'] == 'research']['databases_correct'].sum()
        research_tests = len(df[df['expected_route'] == 'research'])
        
        print(f"Total Tests: {total_tests}")
        print(f"Routing Accuracy: {route_success}/{total_tests} ({route_success/total_tests*100:.1f}%)")
        print(f"Database Selection Accuracy: {db_success}/{research_tests} ({db_success/research_tests*100:.1f}%)")
        
        # Category breakdown
        print("\nResults by Category:")
        category_stats = df.groupby('category').agg({
            'route_correct': ['count', 'sum', 'mean']
        }).round(2)
        print(category_stats)
        
        # Failed tests
        failed_tests = df[~df['route_correct']]
        if len(failed_tests) > 0:
            print("\nFailed Tests:")
            for _, test in failed_tests.iterrows():
                print(f"\n- Query: {test['query']}")
                print(f"  Expected: {test['expected_route']}, Got: {test['actual_route']}")
                print(f"  Reasoning: {test.get('router_reasoning', 'N/A')[:100]}...")
                
        # Database selection analysis
        research_df = df[df['expected_route'] == 'research']
        if len(research_df) > 0:
            print("\nDatabase Selection Analysis:")
            for _, test in research_df.iterrows():
                if not test['databases_correct']:
                    print(f"\n- Query: {test['query']}")
                    print(f"  Expected DBs: {test['expected_databases']}")
                    print(f"  Selected DBs: {test['actual_databases']}")

## Run Tests

In [ ]:
# Initialize test runner
test_runner = AegisTestRunner()

# Run all tests
print("Starting AEGIS routing system tests...\n")
results_df = test_runner.run_all_tests(TEST_CASES)

# Generate report
test_runner.generate_report(results_df)

## Detailed Results Analysis

In [ ]:
# Display detailed results table
display_columns = ['category', 'query', 'expected_route', 'actual_route', 
                   'route_correct', 'expected_databases', 'actual_databases', 
                   'databases_correct']

styled_df = results_df[display_columns].style.apply(
    lambda row: ['background-color: #d4f4dd' if row['route_correct'] else 'background-color: #f4d4d4'] * len(row),
    axis=1
)

display(HTML("<h3>Detailed Test Results</h3>"))
display(styled_df)

## Export Results

In [ ]:
# Save results to file
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
results_file = f"aegis_routing_test_results_{timestamp}.csv"
results_df.to_csv(results_file, index=False)
print(f"Results saved to: {results_file}")

# Save detailed JSON report
json_report = {
    "test_run": timestamp,
    "summary": {
        "total_tests": len(results_df),
        "routing_accuracy": float(results_df['route_correct'].mean()),
        "database_accuracy": float(results_df[results_df['expected_route'] == 'research']['databases_correct'].mean())
    },
    "detailed_results": test_runner.results
}

json_file = f"aegis_routing_test_report_{timestamp}.json"
with open(json_file, 'w') as f:
    json.dump(json_report, f, indent=2)
print(f"Detailed report saved to: {json_file}")

## Custom Test Runner

In [ ]:
# Interactive test runner for custom queries
def test_custom_query(query: str, expected_route: str = None, expected_databases: List[str] = None):
    """Test a custom query"""
    test_case = {
        "category": "Custom",
        "query": query,
        "expected_route": expected_route or "unknown",
        "expected_databases": expected_databases or [],
        "reasoning": "Custom test case"
    }
    
    runner = AegisTestRunner()
    result = runner.run_single_test(test_case)
    
    print(f"Query: {query}")
    print(f"\nClarified: {result.get('clarified_query', 'N/A')}")
    print(f"\nRoute: {result.get('actual_route', 'N/A')}")
    print(f"Router Reasoning: {result.get('router_reasoning', 'N/A')}")
    
    if result.get('actual_route') == 'research':
        print(f"\nDatabases: {result.get('actual_databases', [])}")
        print(f"Planner Reasoning: {result.get('planner_reasoning', 'N/A')}")
    
    return result

# Example usage:
# test_custom_query("What's RBC's dividend yield?", "research", ["public_benchmarking"])